In [1]:
import pandas as pd
import numpy as np
import time
import logging
import os
import warnings
import itertools
from datetime import datetime, timedelta
from functools import partial
from pathlib import Path

seed = 12
np.random.seed(seed)
msg_level = logging.INFO
# Suppress all RuntimeWarnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [2]:
# Create a logger
logger = logging.getLogger("inspect_results_logger")
logger.setLevel(msg_level)  # Set the level for this logger

# Create a handler (where to send the logs)
handler = logging.StreamHandler()  # Send to the console
handler.setLevel(msg_level)

# Create a formatter (how to format the logs)
formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
handler.setFormatter(formatter)

# Add the handler to the logger
logger.addHandler(handler)

In [3]:
benchmark_path = '../data/benchmark_gspc.pkl'
source_path = '../data/stocks_adjclose.pkl'

In [4]:
benchmark = pd.read_pickle(benchmark_path)
benchmark.head()

Ticker,ds,^GSPC
0,2011-01-03,1271.869995
1,2011-01-04,1270.199951
2,2011-01-05,1276.560059
3,2011-01-06,1273.849976
4,2011-01-07,1271.500000


In [5]:
source = pd.read_pickle(source_path)
source.head()

Ticker,A,AAPL,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,ADSK,...,WRB,WST,WTW,WY,WYNN,XEL,XOM,YUM,ZBH,ZBRA
ds,,,,,,,,,,,,,,,,,,,,,
2011-01-03,26.781836,9.917951,16.942663,9.349445,37.585785,31.290001,27.436106,20.712511,29.848969,39.270000,...,6.112024,18.971697,71.104691,11.828708,76.192101,14.700940,43.098133,26.977388,47.841393,38.200001
2011-01-04,26.532440,9.969709,17.102097,9.291334,37.338249,31.510000,27.125227,20.698887,29.741137,38.529999,...,6.072278,18.631702,69.911537,11.702991,78.568939,14.763333,43.300457,26.565216,47.206047,37.840000
2011-01-05,26.474880,10.051261,17.102097,9.305071,37.346004,32.220001,27.183071,20.794275,30.216925,41.240002,...,6.045782,18.713308,70.882263,12.068158,79.582596,14.675976,43.184826,26.691620,47.240875,37.799999
2011-01-06,26.526041,10.043136,17.066668,9.179339,37.485237,32.270000,27.334898,21.591436,30.451658,41.259998,...,5.977333,18.577297,71.084503,11.984348,80.162842,14.663502,43.462337,26.878460,45.778728,37.480000
2011-01-07,26.615564,10.115060,17.137522,9.109608,37.547104,32.040001,27.175838,21.768579,30.521458,40.759998,...,5.939793,18.486635,70.963158,12.313586,83.001091,14.794533,43.699345,27.213682,45.770027,37.599998


In [6]:
df_corr = source.corr()
df_corr.head()

Ticker,A,AAPL,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,ADSK,...,WRB,WST,WTW,WY,WYNN,XEL,XOM,YUM,ZBH,ZBRA
Ticker,,,,,,,,,,,,,,,,,,,,,
A,1.000000,0.938434,0.972057,0.803930,0.973666,0.944446,0.958391,0.877209,0.950709,0.946486,...,0.915526,0.962450,0.952409,0.840455,-0.192150,0.925082,0.581125,0.957106,0.742547,0.926361
AAPL,0.938434,1.000000,0.925958,0.904199,0.965640,0.912658,0.974905,0.858673,0.957237,0.883810,...,0.954041,0.944027,0.944347,0.799052,-0.274004,0.864298,0.709763,0.932155,0.609996,0.834967
ABT,0.972057,0.925958,1.000000,0.780764,0.974008,0.966874,0.947089,0.840192,0.947416,0.967112,...,0.905557,0.957242,0.959083,0.826124,-0.199613,0.959269,0.502779,0.961910,0.787986,0.938066
ACGL,0.803930,0.904199,0.780764,1.000000,0.859009,0.772929,0.921957,0.746998,0.908116,0.731481,...,0.943648,0.783657,0.889515,0.726342,-0.175307,0.746701,0.854873,0.878777,0.539229,0.617688
ACN,0.973666,0.965640,0.974008,0.859009,1.000000,0.962718,0.975225,0.861665,0.975261,0.940212,...,0.947315,0.970085,0.967641,0.856489,-0.227554,0.926912,0.623003,0.971591,0.727454,0.899654


In [7]:
# rank by correlation
corr_sum = df_corr.map(lambda x: abs(x)).sum()
corr_rank = corr_sum.sort_values().rank(method='min').astype(int)
corr_rank

Ticker
TPR       1
BKR       2
GE        3
WYNN      4
DVN       5
       ... 
TXN     435
TEL     436
ITW     437
ICE     438
HD      439
Length: 439, dtype: int64

In [8]:
# rank by returns
return_rank = source.diff().sum(axis=0).sort_values().rank(method='min', ascending=False).astype(int)
return_rank

Ticker
APA     439
MOS     438
SLB     437
DVN     436
PCG     435
       ... 
TDG       5
FICO      4
AZO       3
BKNG      2
NVR       1
Length: 439, dtype: int64

In [9]:
select_10 = (return_rank + corr_rank).sort_values().reset_index()['Ticker'].values[:11]
select_10

array(['FICO', 'LLY', 'MCK', 'CHTR', 'GWW', 'AXON', 'REGN', 'RCL', 'RL',
       'TSLA', 'TPL'], dtype=object)

In [10]:
def generate_data(df, benchmark, days_to_avg=30, days_to_opt=30):
    df2 = df.reset_index()
    benchmark2 = benchmark.reset_index()
    elements = df2.sample(n=100).index # definint a maximum of 100 different sampled initial dates
    for idx in elements:
        df_sample = df2.iloc[idx-days_to_avg:idx+days_to_opt, :]
        df_sample = df_sample.set_index('ds')
        df_sample_b = benchmark2.iloc[idx-days_to_avg:idx+days_to_opt, :]
        df_sample_b = df_sample_b.set_index('ds').drop(['index'], axis=1)
        yield df_sample, df_sample_b

In [11]:
data = source[select_10]

In [12]:
parameters = {
    "population_size": 100,
    "num_generations": 100,
    "mutation_rate": 0.1,
    "elitism": 0.1,
    "n_periods": 10,
    "days_to_avg": 30,
    "days_to_opt": 30,
    "initial_capital": 1000,
}

In [13]:
n_periods = parameters['n_periods']
days_to_avg = parameters['days_to_avg']
days_to_opt = parameters['days_to_opt']
population_size = parameters['population_size']
num_generations = parameters['num_generations']
initial_capital = parameters['initial_capital']

In [14]:
portfolio_value = 10000
portfolio_returns = []
portfolio_total_return = []
portfolio_sharpe_ratios = []
weights_history = pd.DataFrame(index=data.index, columns=data.columns)
portfolio_value_history = pd.Series(index=data.index, name='Portfolio Value', dtype='float')
portfolio_value_history.iloc[0] = portfolio_value


In [15]:
datagen = generate_data(data, benchmark)

In [16]:
df, df_b = next(datagen)

In [17]:
avg_period = days_to_avg
opt_period = days_to_opt

In [18]:
def portfolio_stats(weights, data):

    weights = np.array(weights)
    returns = np.log(data) - np.log(data.shift(1)) # log return to minimize fp error 
    # this function is equivalent to pct_change, but avoids numerical issues with small values
    #returns = data.pct_change().dropna()
    port_return = np.sum(returns.mean() * weights) 
    port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() , weights)))
    try:
        sharpe_ratio = port_return/port_vol
    except Exception as e:
        sharpe_ratio = 0
    return sharpe_ratio, port_return, port_vol

def fitness_function(weights, data):
    sharpe_ratio, _, _ = portfolio_stats(weights, data)
    return sharpe_ratio

In [19]:
import gc
from dimod import Integer, Binary
from dimod import quicksum
from dimod import ConstrainedQuadraticModel, DiscreteQuadraticModel
from dwave.system import LeapHybridDQMSampler, LeapHybridCQMSampler
from dwave.samplers import SimulatedAnnealingSampler, TabuSampler
from dimod import ExactSolver, ExactCQMSolver
from itertools import product

In [25]:
# data is not pct_changea
def compute_risk_and_returns(self, solution):
        """Compute the risk and return values of solution.
        """
        variance = 0.0
        for s1, s2 in product(solution, solution):
            variance += (solution[s1] * self.price[s1] * solution[s2] * self.price[s2] * self.covariance_matrix[s1][s2])

        est_return = 0
        for stock in solution:
            est_return += solution[stock]*self.price[stock]*self.avg_daily_returns[stock]

        return round(est_return, 2), round(variance, 2)

def run(self, min_return=0, max_risk=0):
    return self.solve_cqm(min_return=min_return, max_risk=max_risk)

def dwave_solver(data):
#    solver = ExactCQMSolver()
    solver = SimulatedAnnealingSampler()  # or use LeapHybridCQMSampler for hybrid solver
    cqm = ConstrainedQuadraticModel()
    budget = 1000
    alpha = 0.5
    max_num_shares = (budget/data.iloc[0,:]).astype(int)
    stocks = data.columns.tolist()
    avg_daily_returns = np.log(data) - np.log(data.shift(1)).dropna(axis=0) # log return to minimize fp error 
    x = {s: Integer("%s" %s, lower_bound=0, upper_bound=max_num_shares[s]) for s in stocks}
    df_cov = data.cov()
    risk = 0
    for s1, s2 in product(stocks, stocks):
        coeff = (df_cov[s1][s2] * df[s1] * df[s2])
        risk = risk + coeff * x[s1] * x[s2]

    returns = 0
    for s in stocks:
        returns = returns + df[s] * avg_daily_returns[s] * x[s]

    #cqm.add_constraint(quicksum([x[s]*df[s] for s in stocks]) <= budget, label='upper_budget')
    #cqm.add_constraint(quicksum([x[s]*df[s] for s in stocks]), '>=', rhs=0.997 * budget, label='lower_budget')
    cqm.set_objective(alpha * risk - returns)
    #port_return = np.sum(returns.mean() * weights) 
    #port_vol = np.sqrt(np.dot(weights.T, np.dot(returns.cov() , weights)))
    #sharpe_ratio = port_return/port_vol
    cqm.substitute_self_loops()
    gc.collect()
    sample_set = solver.sample_cqm(cqm)
    n_samples = len(sample_set.record)
    print(f'n_samples: {n_samples}')
    feasible_samples = sample_set.filter(lambda d: d.is_feasible)
    if not feasible_samples:
        raise Exception("No feasible solution could be found for this problem instance.")
    else:
        best_feasible = feasible_samples.first
        solution = {}
        solution['stocks'] = {k:int(best_feasible.sample[k]) for k in stocks}
        print(f'solution_stocks: {solution["stocks"]}')
        solution['return'], solution['risk'] = compute_risk_and_returns(solution['stocks'])
        print(f'solution_return: {solution["return"]}')
        print(f'solution_risk: {solution["risk"]}')
    return np.array(solution['stocks'].values()) / np.sum(solution['stocks'].values())

In [26]:
j = 0
for i in range(avg_period+1, avg_period + opt_period+1):
        df = data.iloc[j:i, :]
        df_pct = df.iloc[j:i, :].pct_change().dropna(axis=0)
        #logger.debug(f'df_pct: {df_pct}')
        budget = 1000
        weights = dwave_solver(df)
        break


TypeError: 'QuadraticModel' object is not iterable

## Examples from D-wave's website

In [ ]:
from dimod.generators import and_gate
from dimod import ExactSolver
bqm = and_gate('in1', 'in2', 'out')
sampler = ExactSolver()
sampleset = sampler.sample(bqm)
print(sampleset)       

  in1 in2 out energy num_oc.
0   0   0   0    0.0       1
1   1   0   0    0.0       1
3   0   1   0    0.0       1
5   1   1   1    0.0       1
2   1   1   0    1.0       1
4   0   1   1    1.0       1
6   1   0   1    1.0       1
7   0   0   1    3.0       1
['BINARY', 8 rows, 8 samples, 3 variables]


In [ ]:
from dwave.samplers import SimulatedAnnealingSampler
solver = SimulatedAnnealingSampler()
sampleset = solver.sample_ising({'a': -0.5, 'b': 1.0}, {('a', 'b'): -1}, num_reads=10)
sampleset.first.sample["a"] == sampleset.first.sample["b"] == -1

np.True_

In [ ]:
import networkx as nx
import dimod
from dwave.samplers import SimulatedAnnealingSampler, TabuSampler
G = nx.generators.les_miserables_graph()
bqm = dimod.generators.maximum_independent_set(G.edges, G.nodes)
len(bqm)
sampleset_sa = SimulatedAnnealingSampler().sample(bqm, num_reads=10)
sampleset_tabu = TabuSampler().sample(bqm, num_reads=100)
sum(sampleset_sa.first.sample.values())                    
sum(sampleset_tabu.first.sample.values())                  
[key for key, val in sampleset_sa.first.sample.items() if val][0:5] 

['Anzelma', 'BaronessT', 'Boulatruelle', 'Champtercier', 'Child2']